<a href="https://colab.research.google.com/github/khiemp110/template/blob/main/colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## KoboldCpp + StableUI trên Colab

1. Chọn **GPU runtime**, bật `UsePreset`, chọn `Template` rồi chạy cell bên dưới. Tắt `UsePreset` để cấu hình thủ công.
2. Bật `AllowSaveToGoogleDrive` để lưu và dùng lại model trong **MyDrive/Colab Notebooks/KoboldCPP/models/**. Model được tải bằng mạng **Colab**.
3. Mở link tunnel, thêm **`/sdui/`** để vào StableUI. Bấm **Stop** để dừng.

`DeleteExistingModels` chỉ xóa cache tạm trên Colab. `CLOUDFLARE_API_KEY` (Colab Secret) dùng named tunnel đã route tới `http://localhost:5001`; nếu không có, dùng tunnel mặc định.


In [ ]:
#@title Âm thanh tùy chọn (không đảm bảo giữ runtime) { display-mode: "form" }
%%html
<audio src="https://raw.githubusercontent.com/KoboldAI/KoboldAI-Client/main/colab/silence.m4a" loop controls></audio>

In [ ]:
# @title <b>v-- Chọn Model bên dưới rồi bấm Play để chạy KoboldCpp nhé peko!</b> { display-mode: "form" }
# @markdown Nhớ bấm nút Play khởi động lại mỗi khi ngươi đổi cấu hình đó!

from contextlib import contextmanager
import hashlib
import json
import os
from pathlib import Path
import shlex
import signal
import shutil
import subprocess
import sys
import threading
import time
from urllib.parse import unquote, urlparse
from google.colab import drive, userdata

# @markdown ### 1. Chọn nhanh Preset có sẵn (Tùy chọn)
UsePreset = True # @param {type:"boolean"}
Template = "Qwen Image 2.1 Max (Q8 L4)" # @param ["Gemma4 E4B Uncensored (General)", "Tiefighter 13B (General)", "Estopia 13B (General)", "Qwen3 VL 8B (Vision)", "Cydonia 24B (Roleplay)", "Broken Tutu 24B (Roleplay)", "Dans PersonalityEngine 24B (Roleplay)", "Harbinger 24B (Instruct)", "Muse 12B (Instruct)", "Mistral Small 24B (Corporate)", "GPT OSS 20B (Corporate)", "Gemma3 12B (Corporate)", "GLM4 9b chat (Corporate)", "Z-Image Turbo (Image)", "Flux Klein 4B (Image)", "Qwen Image 2.1 (Image)", "Qwen Image 2.1 Max (Q8 L4)"]

# @markdown ### 2. Cấu hình thủ công (chỉ áp dụng khi tắt UsePreset)
# @markdown <hr>
LoadTextModel = False # @param {type:"boolean"}
Model = "Gemma 4 12B (Q4_K_M)" # @param ["Gemma 4 12B (Q4_K_M)", "Gemma 4 E4B Uncensored (Q5_K_M)", "Tiefighter 13B (Q4_K_S)", "Estopia 13B (Q4_K_S)", "Llama 3.1 8B BookAdventures (Q6_K)", "Cydonia 24B v4.2.0 (Q4_K_S)", "Broken Tutu 24B (Q4_K_S)", "Dans PersonalityEngine 24B (Q4_K_S)", "Harbinger 24B (Q4_K_S)", "Muse 12B (Q4_K_S)", "Qwen3 VL 8B Instruct (Q6_K)", "Mistral Small 3.2 24B (Q4_K_S)", "GPT OSS 20B (MXFP4)", "Gemma 3 12B (Q4_K_S)", "Gemma 3n E4B (Q6_K)", "GLM 4 9B (Q6_K)"] {allow-input: true}
Layers = "Auto" # @param ["Auto","999"] {allow-input: true}
ContextSize = "40960" # @param ["4096","8192","12288","16384","24576","32768","40960"] {allow-input: true}

# @markdown <hr>
LoadVisionMMProjector = False # @param {type:"boolean"}
Mmproj = "Gemma 4 E4B mmproj (BF16)" # @param ["Gemma 4 E4B mmproj (BF16)", "Qwen3 VL 8B mmproj (BF16)", "LLaMA 13B mmproj v1.5 (Q4_1)", "LLaMA 3 8B mmproj (Q4_1)"] {allow-input: true}

# @markdown <hr>
LoadImgModel = True # @param {type:"boolean"}
ImgModel = "Qwen Image 2.1 (Q8_0 - Max Quality)" # @param ["Qwen Image 2.1 (Q8_0 - Max Quality)", "Qwen Image 2.1 (Q5_K_M)", "PicX Real (Q8_0)", "DreamShaper XL v2 (Q4_K)", "Flux Klein 4B (Q4_0)", "Flux Klein 9B (Q5_K_M)"] {allow-input: true}
ImgClip1 = "Qwen3 VL 8B Instruct (Q8_0 - Max Text)" # @param ["Qwen3 VL 8B Instruct (Q8_0 - Max Text)", "Qwen3 VL 8B Instruct (UD-Q4_K_XL - Standard)", "Qwen3 4B (Q4_K_M - Low VRAM)"] {allow-input: true}
ImgVAE = "Qwen Image 2.1 VAE (Texture-Fix BF16)" # @param ["Qwen Image 2.1 VAE (Texture-Fix BF16)", "Flux2 VAE"] {allow-input: true}

# @markdown <hr>
LoadSpeechModel = False # @param {type:"boolean"}
SpeechModel = "Whisper Base EN (Q5_1)" # @param ["Whisper Base EN (Q5_1)"] {allow-input: true}

# @markdown <hr>
LoadTTSModel = False # @param {type:"boolean"}
TTSModel = "Kokoro (Q4)" # @param ["Kokoro (Q4)", "OuteTTS 0.2 500M (Q4_0)"] {allow-input: true}
WavTokModel = "None" # @param ["None", "WavTokenizer Large 75 (Q4_0)"] {allow-input: true}

# @markdown <hr>
LoadEmbeddingsModel = False # @param {type:"boolean"}
EmbeddingsModel = "Snowflake Arctic Embed S (Q4_0)" # @param ["Snowflake Arctic Embed S (Q4_0)"] {allow-input: true}

# @markdown <hr>
LoadMusicModel = False # @param {type:"boolean"}
MusicArgs = "--musiclowvram --musicllm https://huggingface.co/koboldcpp/music/resolve/main/acestep-5Hz-lm-1.7B-BF16.gguf --musicembeddings https://huggingface.co/koboldcpp/music/resolve/main/Ace-Qwen3-Embedding-0.6B-BF16.gguf --musicdiffusion https://huggingface.co/koboldcpp/music/resolve/main/acestep-v15-sftturbo50-Q8_0.gguf --musicvae https://huggingface.co/koboldcpp/music/resolve/main/ace-vae-BF16.gguf"

# @markdown <hr>
# @markdown ### 3. Tiện ích hệ thống & Tunnel
# @markdown DeleteExistingModels chỉ xóa cache tạm /content/koboldcpp-cache, không xóa model trên Drive.
DeleteExistingModels = False # @param {type:"boolean"}
Multiplayer = False # @param {type:"boolean"}
# @markdown Lưu và dùng lại model tại MyDrive/Colab Notebooks/KoboldCPP; Colab tự tải bằng mạng của runtime.
AllowSaveToGoogleDrive = True # @param {type:"boolean"}
MakeLocalTunnelFallback = False # @param {type:"boolean"}

# --- TỪ ĐIỂN ÁNH XẠ TÊN ĐẸP -> LINK HUGGINGFACE ---
MAP_TEXT_MODELS = {
    "Gemma 4 12B (Q4_K_M)": "https://huggingface.co/unsloth/gemma-4-12b-it-GGUF/resolve/main/gemma-4-12b-it-Q4_K_M.gguf",
    "Gemma 4 E4B Uncensored (Q5_K_M)": "https://huggingface.co/HauhauCS/Gemma-4-E4B-Uncensored-HauhauCS-Aggressive/resolve/main/Gemma-4-E4B-Uncensored-HauhauCS-Aggressive-Q5_K_M.gguf",
    "Tiefighter 13B (Q4_K_S)": "https://huggingface.co/KoboldAI/LLaMA2-13B-Tiefighter-GGUF/resolve/main/LLaMA2-13B-Tiefighter.Q4_K_S.gguf",
    "Estopia 13B (Q4_K_S)": "https://huggingface.co/KoboldAI/LLaMA2-13B-Estopia-GGUF/resolve/main/LLaMA2-13B-Estopia.Q4_K_S.gguf",
    "Llama 3.1 8B BookAdventures (Q6_K)": "https://huggingface.co/KoboldAI/Llama-3.1-8B-BookAdventures-GGUF/resolve/main/Llama-3.1-8B-BookAdventures.Q6_K.gguf",
    "Cydonia 24B v4.2.0 (Q4_K_S)": "https://huggingface.co/bartowski/TheDrummer_Cydonia-24B-v4.2.0-GGUF/resolve/main/TheDrummer_Cydonia-24B-v4.2.0-Q4_K_S.gguf",
    "Broken Tutu 24B (Q4_K_S)": "https://huggingface.co/mradermacher/Broken-Tutu-24B-GGUF/resolve/main/Broken-Tutu-24B.Q4_K_S.gguf",
    "Dans PersonalityEngine 24B (Q4_K_S)": "https://huggingface.co/bartowski/PocketDoc_Dans-PersonalityEngine-V1.3.0-24b-GGUF/resolve/main/PocketDoc_Dans-PersonalityEngine-V1.3.0-24b-Q4_K_S.gguf",
    "Harbinger 24B (Q4_K_S)": "https://huggingface.co/LatitudeGames/Harbinger-24B-GGUF/resolve/main/Harbinger-24B-Q4_K_S.gguf",
    "Muse 12B (Q4_K_S)": "https://huggingface.co/LatitudeGames/Muse-12B-GGUF/resolve/main/Muse-12B-Q4_K_S.gguf",
    "Qwen3 VL 8B Instruct (Q6_K)": "https://huggingface.co/unsloth/Qwen3-VL-8B-Instruct-GGUF/resolve/main/Qwen3-VL-8B-Instruct-Q6_K.gguf",
    "Mistral Small 3.2 24B (Q4_K_S)": "https://huggingface.co/unsloth/Mistral-Small-3.2-24B-Instruct-2506-GGUF/resolve/main/Mistral-Small-3.2-24B-Instruct-2506-Q4_K_S.gguf",
    "GPT OSS 20B (MXFP4)": "https://huggingface.co/ggml-org/gpt-oss-20b-GGUF/resolve/main/gpt-oss-20b-mxfp4.gguf",
    "Gemma 3 12B (Q4_K_S)": "https://huggingface.co/bartowski/google_gemma-3-12b-it-GGUF/resolve/main/google_gemma-3-12b-it-Q4_K_S.gguf",
    "Gemma 3n E4B (Q6_K)": "https://huggingface.co/unsloth/gemma-3n-E4B-it-GGUF/resolve/main/gemma-3n-E4B-it-Q6_K.gguf",
    "GLM 4 9B (Q6_K)": "https://huggingface.co/unsloth/GLM-4-9B-0414-GGUF/resolve/main/GLM-4-9B-0414-Q6_K.gguf"
}

MAP_MMPROJ = {
    "Gemma 4 E4B mmproj (BF16)": "https://huggingface.co/unsloth/gemma-4-E4B-it-GGUF/resolve/main/mmproj-BF16.gguf",
    "Qwen3 VL 8B mmproj (BF16)": "https://huggingface.co/unsloth/Qwen3-VL-8B-Instruct-GGUF/resolve/main/mmproj-BF16.gguf",
    "LLaMA 13B mmproj v1.5 (Q4_1)": "https://huggingface.co/koboldcpp/mmproj/resolve/main/llama-13b-mmproj-v1.5.Q4_1.gguf",
    "LLaMA 3 8B mmproj (Q4_1)": "https://huggingface.co/koboldcpp/mmproj/resolve/main/LLaMA3-8B_mmproj-Q4_1.gguf"
}

MAP_IMG_MODELS = {
    "Qwen Image 2.1 (Q8_0 - Max Quality)": "https://huggingface.co/unsloth/Qwen-Image-2.1-GGUF/resolve/main/qwen-image-2.1-Q8_0.gguf",
    "Qwen Image 2.1 (Q5_K_M)": "https://huggingface.co/unsloth/Qwen-Image-2.1-GGUF/resolve/main/qwen-image-2.1-Q5_K_M.gguf",
    "PicX Real (Q8_0)": "https://huggingface.co/koboldcpp/imgmodel/resolve/main/picx_real_q8_0.gguf",
    "DreamShaper XL v2 (Q4_K)": "https://huggingface.co/koboldcpp/imgmodel/resolve/main/dreamshaperXL_v2_q4_k.gguf",
    "Flux Klein 4B (Q4_0)": "https://huggingface.co/leejet/FLUX.2-klein-4B-GGUF/resolve/main/flux-2-klein-4b-Q4_0.gguf",
    "Flux Klein 9B (Q5_K_M)": "https://huggingface.co/unsloth/FLUX.2-klein-9B-GGUF/resolve/main/flux-2-klein-9b-Q5_K_M.gguf"
}

MAP_IMG_CLIPS = {
    "Qwen3 VL 8B Instruct (Q8_0 - Max Text)": "https://huggingface.co/unsloth/Qwen3-VL-8B-Instruct-GGUF/resolve/main/Qwen3-VL-8B-Instruct-Q8_0.gguf",
    "Qwen3 VL 8B Instruct (UD-Q4_K_XL - Standard)": "https://huggingface.co/unsloth/Qwen3-VL-8B-Instruct-GGUF/resolve/main/Qwen3-VL-8B-Instruct-UD-Q4_K_XL.gguf",
    "Qwen3 4B (Q4_K_M - Low VRAM)": "https://huggingface.co/unsloth/Qwen3-4B-GGUF/resolve/main/Qwen3-4B-Q4_K_M.gguf"
}

MAP_IMG_VAES = {
    "Qwen Image 2.1 VAE (Texture-Fix BF16)": "https://huggingface.co/madebyollin/texture-fix-vae-for-qwen-image-2.1/resolve/main/texture_fix_vae_for_qwen_image_2.1_bf16.safetensors",
    "Flux2 VAE": "https://huggingface.co/koboldcpp/GGUFDumps/resolve/main/flux2vae.safetensors"
}

MAP_SPEECH_MODELS = {
    "Whisper Base EN (Q5_1)": "https://huggingface.co/koboldcpp/whisper/resolve/main/whisper-base.en-q5_1.bin"
}

MAP_TTS_MODELS = {
    "Kokoro (Q4)": "https://huggingface.co/koboldcpp/tts/resolve/main/Kokoro_no_espeak_Q4.gguf",
    "OuteTTS 0.2 500M (Q4_0)": "https://huggingface.co/koboldcpp/tts/resolve/main/OuteTTS-0.2-500M-Q4_0.gguf"
}

MAP_WAVTOK_MODELS = {
    "None": "",
    "WavTokenizer Large 75 (Q4_0)": "https://huggingface.co/koboldcpp/tts/resolve/main/WavTokenizer-Large-75-Q4_0.gguf"
}

MAP_EMB_MODELS = {
    "Snowflake Arctic Embed S (Q4_0)": "https://huggingface.co/yixuan-chia/snowflake-arctic-embed-s-GGUF/resolve/main/snowflake-arctic-embed-s-Q4_0.gguf"
}

def set_log_path(path, drive_root=None):
    global SESSION_LOG_PATH, SESSION_LOG_STARTED, SESSION_DRIVE_LOG_PATH
    SESSION_LOG_PATH = Path(path)
    SESSION_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    SESSION_LOG_STARTED = time.monotonic()
    SESSION_DRIVE_LOG_PATH = (Path(drive_root) / "logs" / f"session-{time.time_ns()}.log"
                             if drive_root is not None else None)
    log_message(f"Log trực tiếp trong cell; file đầy đủ: {SESSION_LOG_PATH}")


def log_message(message):
    started = globals().get("SESSION_LOG_STARTED")
    prefix = f"[+{time.monotonic() - started:.1f}s] " if started is not None else ""
    line = prefix + str(message).rstrip("\r\n")
    print(line, flush=True)
    path = globals().get("SESSION_LOG_PATH")
    if path is not None:
        with Path(path).open("a", encoding="utf-8") as stream:
            stream.write(line + "\n")


def save_log():
    source = globals().get("SESSION_LOG_PATH")
    target = globals().get("SESSION_DRIVE_LOG_PATH")
    if source is not None and target is not None:
        try:
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, target)
        except OSError as error:
            print(f"Không sao lưu được log lên Drive: {error}", flush=True)


@contextmanager
def command_stage(label, heartbeat_seconds=15):
    started = time.monotonic()
    stopped = threading.Event()
    log_message(f"BẮT ĐẦU: {label}")

    def heartbeat():
        while not stopped.wait(heartbeat_seconds):
            log_message(f"{label}: đang chạy, đã {time.monotonic() - started:.0f}s; chưa kết thúc bước này.")

    ticker = threading.Thread(target=heartbeat, daemon=True)
    ticker.start()
    try:
        yield
    except BaseException as error:
        log_message(f"DỪNG/LỖI: {label} sau {time.monotonic() - started:.1f}s ({type(error).__name__}: {error})")
        raise
    else:
        log_message(f"XONG: {label} sau {time.monotonic() - started:.1f}s")
    finally:
        stopped.set()
        ticker.join(timeout=1)


def run_logged_command(arguments, *, cwd=None, label=None, heartbeat_seconds=15):
    """Pipe child output through Python so Colab actually displays it in the cell."""
    command = [str(argument) for argument in arguments]
    label = label or Path(command[0]).name
    log_message(f"Lệnh: {shlex.join(command)}")
    if cwd is not None:
        log_message(f"Thư mục: {cwd}")
    environment = {**os.environ, "PYTHONUNBUFFERED": "1"}
    # Also flush C/C++ stdout on Colab; Python's bufsize only affects the reader.
    executable = command
    if sys.platform == "linux" and shutil.which("stdbuf"):
        executable = ["stdbuf", "-oL", "-eL", *command]
    process = None
    try:
        with command_stage(label, heartbeat_seconds):
            process = subprocess.Popen(executable, cwd=cwd, stdout=subprocess.PIPE,
                                       stderr=subprocess.STDOUT, text=True, errors="replace",
                                       bufsize=1, env=environment, start_new_session=True)
            log_message(f"{label}: PID {process.pid}")
            for line in process.stdout:
                log_message(line)
            returncode = process.wait()
            log_message(f"{label}: exit code {returncode}")
            if returncode:
                log_message(f"Xem toàn bộ stdout/stderr tại: {globals().get('SESSION_LOG_PATH', 'output của cell')}")
                raise subprocess.CalledProcessError(returncode, command)
            return subprocess.CompletedProcess(command, returncode)
    finally:
        if process is not None:
            if process.poll() is None:
                try:
                    os.killpg(process.pid, signal.SIGTERM)
                except ProcessLookupError:
                    pass
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    os.killpg(process.pid, signal.SIGKILL)
                    process.wait(timeout=5)
            process.stdout.close()
        save_log()


def sha256_file(path, *, label=None):
    path = Path(path)
    label = label or f"SHA256 {path.name}"
    digest = hashlib.sha256()
    total = path.stat().st_size
    done = 0
    last_report = time.monotonic()
    if label:
        log_message(f"{label}: đọc {total / 1e9:.2f} GB để kiểm tra SHA256")
    with path.open("rb") as stream:
        while chunk := stream.read(8 * 1024 * 1024):
            digest.update(chunk)
            done += len(chunk)
            if label and time.monotonic() - last_report >= 10:
                log_message(f"{label}: {done / 1e9:.2f}/{total / 1e9:.2f} GB ({100 * done / total:.1f}%)")
                last_report = time.monotonic()
    if label:
        log_message(f"{label}: đã đọc xong {total / 1e9:.2f} GB")
    return digest.hexdigest()

PRESETS = {
    "Gemma4 E4B Uncensored (General)": {
        "model": "https://huggingface.co/HauhauCS/Gemma-4-E4B-Uncensored-HauhauCS-Aggressive/resolve/main/Gemma-4-E4B-Uncensored-HauhauCS-Aggressive-Q5_K_M.gguf",
        "ctx": "40960",
        "mmproj": "https://huggingface.co/unsloth/gemma-4-E4B-it-GGUF/resolve/main/mmproj-BF16.gguf"
    },
    "Tiefighter 13B (General)": {
        "model": "https://huggingface.co/KoboldAI/LLaMA2-13B-Tiefighter-GGUF/resolve/main/LLaMA2-13B-Tiefighter.Q4_K_S.gguf",
        "ctx": "4096",
        "mmproj": "https://huggingface.co/koboldcpp/mmproj/resolve/main/llama-13b-mmproj-v1.5.Q4_1.gguf"
    },
    "Estopia 13B (General)": {
        "model": "https://huggingface.co/KoboldAI/LLaMA2-13B-Estopia-GGUF/resolve/main/LLaMA2-13B-Estopia.Q4_K_S.gguf",
        "ctx": "4096",
        "mmproj": "https://huggingface.co/koboldcpp/mmproj/resolve/main/llama-13b-mmproj-v1.5.Q4_1.gguf"
    },
    "Qwen3 VL 8B (Vision)": {
        "model": "https://huggingface.co/unsloth/Qwen3-VL-8B-Instruct-GGUF/resolve/main/Qwen3-VL-8B-Instruct-Q6_K.gguf",
        "ctx": "32768",
        "mmproj": "https://huggingface.co/unsloth/Qwen3-VL-8B-Instruct-GGUF/resolve/main/mmproj-BF16.gguf"
    },
    "Cydonia 24B (Roleplay)": {"model": "https://huggingface.co/bartowski/TheDrummer_Cydonia-24B-v4.2.0-GGUF/resolve/main/TheDrummer_Cydonia-24B-v4.2.0-Q4_K_S.gguf", "ctx": "10240"},
    "Broken Tutu 24B (Roleplay)": {"model": "https://huggingface.co/mradermacher/Broken-Tutu-24B-GGUF/resolve/main/Broken-Tutu-24B.Q4_K_S.gguf", "ctx": "10240"},
    "Dans PersonalityEngine 24B (Roleplay)": {"model": "https://huggingface.co/bartowski/PocketDoc_Dans-PersonalityEngine-V1.2.0-24b-GGUF/resolve/main/PocketDoc_Dans-PersonalityEngine-V1.2.0-24b-IQ4_XS.gguf", "ctx": "10240"},
    "Harbinger 24B (Instruct)": {"model": "https://huggingface.co/LatitudeGames/Harbinger-24B-GGUF/resolve/main/Harbinger-24B-Q4_K_S.gguf", "ctx": "10240"},
    "Muse 12B (Instruct)": {"model": "https://huggingface.co/LatitudeGames/Muse-12B-GGUF/resolve/main/Muse-12B-Q4_K_S.gguf", "ctx": "49152"},
    "Mistral Small 24B (Corporate)": {"model": "https://huggingface.co/unsloth/Mistral-Small-3.2-24B-Instruct-2506-GGUF/resolve/main/Mistral-Small-3.2-24B-Instruct-2506-Q4_K_S.gguf", "ctx": "10240"},
    "GPT OSS 20B (Corporate)": {"model": "https://huggingface.co/ggml-org/gpt-oss-20b-GGUF/resolve/main/gpt-oss-20b-mxfp4.gguf", "ctx": "32768"},
    "Gemma3 12B (Corporate)": {"model": "https://huggingface.co/bartowski/google_gemma-3-12b-it-GGUF/resolve/main/google_gemma-3-12b-it-Q4_K_S.gguf", "ctx": "16384"},
    "GLM4 9b chat (Corporate)": {"model": "https://huggingface.co/unsloth/GLM-4-9B-0414-GGUF/resolve/main/GLM-4-9B-0414-Q6_K.gguf", "ctx": "49152"},
    "Z-Image Turbo (Image)": {
        "img_model": "https://huggingface.co/leejet/Z-Image-Turbo-GGUF/resolve/main/z_image_turbo-Q4_0.gguf",
        "img_clip": "https://huggingface.co/unsloth/Qwen3-4B-Instruct-2507-GGUF/resolve/main/Qwen3-4B-Instruct-2507-Q4_K_S.gguf",
        "img_vae": "https://huggingface.co/camenduru/FLUX.1-dev/resolve/main/ae.safetensors"
    },
    "Flux Klein 4B (Image)": {
        "img_model": "https://huggingface.co/leejet/FLUX.2-klein-4B-GGUF/resolve/main/flux-2-klein-4b-Q4_0.gguf",
        "img_clip": "https://huggingface.co/unsloth/Qwen3-4B-GGUF/resolve/main/Qwen3-4B-Q4_K_M.gguf",
        "img_vae": "https://huggingface.co/koboldcpp/GGUFDumps/resolve/main/flux2vae.safetensors"
    },
    "Qwen Image 2.1 Max (Q8 L4)": {
        "img_model": "https://huggingface.co/unsloth/Qwen-Image-2.1-GGUF/resolve/main/qwen-image-2.1-Q8_0.gguf",
        "img_clip": "https://huggingface.co/unsloth/Qwen3-VL-8B-Instruct-GGUF/resolve/main/Qwen3-VL-8B-Instruct-Q8_0.gguf",
        "img_vae": "https://huggingface.co/madebyollin/texture-fix-vae-for-qwen-image-2.1/resolve/main/texture_fix_vae_for_qwen_image_2.1_bf16.safetensors"
    },
    "Qwen Image 2.1 (Image)": {
        "img_model": "https://huggingface.co/unsloth/Qwen-Image-2.1-GGUF/resolve/main/qwen-image-2.1-Q5_K_M.gguf",
        "img_clip": "https://huggingface.co/unsloth/Qwen3-VL-8B-Instruct-GGUF/resolve/main/Qwen3-VL-8B-Instruct-UD-Q4_K_XL.gguf",
        "img_vae": "https://huggingface.co/madebyollin/texture-fix-vae-for-qwen-image-2.1/resolve/main/texture_fix_vae_for_qwen_image_2.1_bf16.safetensors"
    }
}


def configure_models(use_preset, template, manual):
    if not use_preset:
        return dict(manual)
    if template not in PRESETS:
        raise ValueError(f"Preset không tồn tại: {template}")
    print(f"Áp dụng preset: {template}")
    return dict(PRESETS[template])


def download_file(url, dest_dir=None, custom_name=None, *, expected_size=None, expected_sha256=None):
    """Runs inside Colab: resume into .part and expose only completed downloads."""
    if not url:
        return None
    parsed = urlparse(url)
    if parsed.scheme != "https" or not parsed.netloc:
        raise ValueError("Model phải là URL HTTPS trực tiếp tới file.")
    filename = custom_name or unquote(Path(parsed.path).name)
    if not filename or Path(filename).name != filename or filename in {".", ".."}:
        raise ValueError("Tên file không hợp lệ.")
    # Distinguish repositories/revisions even when both files are named mmproj.gguf.
    identity = hashlib.sha256(url.encode()).hexdigest()[:16]
    directory = Path(dest_dir or MODEL_CACHE_DIR) / identity
    directory.mkdir(parents=True, exist_ok=True)
    target = directory / filename
    receipt = directory / (filename + ".complete.json")
    partial = directory / (filename + ".part")
    control = directory / (filename + ".part.aria2")

    if target.exists():
        record = json.loads(receipt.read_text()) if receipt.exists() else {}
        known = expected_size is not None and expected_sha256 is not None
        if not known and (record.get("url") != url or not record.get("sha256")):
            raise RuntimeError(f"File chưa có xác nhận tải xong: {target}; kiểm tra hoặc chuyển file đi rồi chạy lại.")
        size = expected_size if known else record["size"]
        digest = expected_sha256 if known else record["sha256"]
        print(f"Kiểm tra cache: {filename} ...", flush=True)
        if target.stat().st_size != size or sha256_file(target) != digest:
            raise RuntimeError(f"Cache sai size/SHA256: {target}; không ghi đè. Chuyển file lỗi đi rồi chạy lại.")
        print(f"Dùng lại model trên {'Drive' if AllowSaveToGoogleDrive else 'runtime'}: {filename}", flush=True)
        return str(target)

    downloaded = partial.stat().st_size if partial.exists() else 0
    if expected_size is not None:
        if downloaded > expected_size:
            raise RuntimeError(f"File tải dở lớn hơn model: {partial}")
        if shutil.disk_usage(directory).free < expected_size - downloaded:
            raise RuntimeError(f"Không đủ dung lượng cho {filename}.")
    print(f"Colab tải → {target} (tiếp tục từ file tải dở nếu có)", flush=True)
    run_logged_command([
        "aria2c", "--console-log-level=notice", "--summary-interval=10",
        "--continue=true", "--auto-file-renaming=false", "--allow-overwrite=true",
        "--file-allocation=none", "--max-tries=5", "--retry-wait=5",
        "--connect-timeout=30", "--timeout=120", "-x", "8", "-s", "8", "-k", "1M",
        "--dir", str(directory), "--out", partial.name, url,
    ], label=f"Tải model {filename}")
    if not partial.is_file() or partial.stat().st_size == 0 or control.exists():
        raise RuntimeError(f"Tải chưa hoàn tất: {partial}. Chạy lại để tiếp tục.")
    if expected_size is not None and partial.stat().st_size != expected_size:
        raise RuntimeError(f"Sai kích thước: {partial}; không sử dụng file này.")
    print(f"Kiểm tra SHA256: {filename} ...", flush=True)
    digest = sha256_file(partial)
    if expected_sha256 is not None and digest != expected_sha256:
        raise RuntimeError(f"Sai SHA256: {partial}; chuyển file lỗi đi trước khi tải lại.")
    record = {"url": url, "size": partial.stat().st_size, "sha256": digest}
    target_temporary_receipt = receipt.with_suffix(".json.tmp")
    target_temporary_receipt.write_text(json.dumps(record), encoding="utf-8")
    partial.rename(target)
    target_temporary_receipt.replace(receipt)
    return str(target)


def download_binary(url, target):
    target = Path(target)
    if not target.is_file() or not os.access(target, os.X_OK):
        partial = target.with_suffix(".part")
        run_logged_command(["curl", "--fail", "--location", "--retry", "3",
                            "--output", str(partial), url], label=f"Tải {target.name}")
        partial.chmod(0o755)
        partial.replace(target)
    return str(target)


def build_kobold_command(config):
    binary = download_binary("https://github.com/LostRuins/koboldcpp/releases/latest/download/koboldcpp-linux-x64",
                             "/content/koboldcpp-linux-x64")
    layers = -1 if str(Layers).lower() == "auto" else int(Layers)
    command = [binary, "--gpulayers", str(layers), "--contextsize", str(config.get("ctx", ContextSize)), "--usecublas"]
    cwd = "/content"
    for key, flag, mapping in (
        ("model", "--model", MAP_TEXT_MODELS), ("mmproj", "--mmproj", MAP_MMPROJ),
        ("img_model", "--sdmodel", MAP_IMG_MODELS), ("img_clip", "--sdllm", MAP_IMG_CLIPS),
        ("img_vae", "--sdvae", MAP_IMG_VAES), ("speech", "--whispermodel", MAP_SPEECH_MODELS),
        ("tts", "--ttsmodel", MAP_TTS_MODELS), ("wavtok", "--ttswavtokenizer", MAP_WAVTOK_MODELS),
        ("embeddings", "--embeddingsmodel", MAP_EMB_MODELS),
    ):
        value = config.get(key)
        if value:
            url = mapping.get(value, value)
            if url:
                path = download_file(url)
                command.extend([flag, path])
    if config.get("music"):
        music = shlex.split(config["music"])
        for index, arg in enumerate(music[:-1]):
            if arg in {"--musicllm", "--musicembeddings", "--musicdiffusion", "--musicvae"}:
                music[index + 1] = download_file(music[index + 1])
        command.extend(music)
    if Multiplayer:
        command.append("--multiplayer")
    return command, cwd


def run_server(command, cwd):
    return run_logged_command(command, cwd=cwd, label="KoboldCpp / StableUI")


manual = {
    "model": Model if LoadTextModel else None,
    "mmproj": Mmproj if LoadVisionMMProjector else None,
    "img_model": ImgModel if LoadImgModel else None,
    "img_clip": ImgClip1 if LoadImgModel else None,
    "img_vae": ImgVAE if LoadImgModel else None,
    "speech": SpeechModel if LoadSpeechModel else None,
    "tts": TTSModel if LoadTTSModel else None,
    "wavtok": WavTokModel if LoadTTSModel else None,
    "embeddings": EmbeddingsModel if LoadEmbeddingsModel else None,
    "music": MusicArgs if LoadMusicModel else None,
}
config = configure_models(UsePreset, Template, manual)
LOCAL_CACHE_ROOT = Path("/content/koboldcpp-cache")
if DeleteExistingModels and LOCAL_CACHE_ROOT.exists():
    shutil.rmtree(LOCAL_CACHE_ROOT)
    print("Đã xóa cache tạm của notebook trong /content. Model trên Drive được giữ lại.")
if AllowSaveToGoogleDrive:
    mount = Path("/content/drive")
    if not (mount.is_mount() and (mount / "MyDrive").is_dir()):
        drive.mount(str(mount))
    if not (mount / "MyDrive").is_dir():
        raise RuntimeError("Drive chưa sẵn sàng; hoàn tất cấp quyền Google rồi chạy lại.")
    CACHE_ROOT = mount / "MyDrive/Colab Notebooks/KoboldCPP"
else:
    CACHE_ROOT = LOCAL_CACHE_ROOT
MODEL_CACHE_DIR = CACHE_ROOT / "models"
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Cache model: {MODEL_CACHE_DIR}. Tải và đọc model đều do máy Colab thực hiện.")
set_log_path("/content/koboldcpp-logs/session.log", CACHE_ROOT if AllowSaveToGoogleDrive else None)
if shutil.which("aria2c") is None:
    run_logged_command(["apt-get", "update", "-qq"], label="Cập nhật danh sách gói")
    run_logged_command(["apt-get", "install", "-y", "-qq", "aria2"], label="Cài aria2")
cmd_args, working_directory = build_kobold_command(config)

cf_token = ""
try:
    cf_token = (userdata.get("CLOUDFLARE_API_KEY") or "").strip()
except Exception:
    pass

# The custom Cloudflare tunnel must route to http://localhost:5001.
tunnels = []
try:
    if cf_token:
        cloudflared = download_binary("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
                                     "/content/cloudflared")
        print("Mở Cloudflare tunnel đã cấu hình tới localhost:5001.")
        tunnels.append(subprocess.Popen([cloudflared, "tunnel", "run", "--token", cf_token],
                                         stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT))
    else:
        cmd_args.append("--remotetunnel")
    if MakeLocalTunnelFallback:
        run_logged_command(["npm", "install", "-g", "localtunnel"], label="Cài Localtunnel")
        tunnels.append(subprocess.Popen(["lt", "--port", "5001"]))
        print("Mật khẩu Localtunnel:", flush=True)
        run_logged_command(["curl", "--fail", "--silent", "https://localtunnel.github.io/bypass"], label="Địa chỉ Localtunnel")
    print(f"\nKhởi động KoboldCpp:\n{shlex.join(cmd_args)}", flush=True)
    print("StableUI: thêm /sdui/ vào cuối link tunnel của KoboldCpp.", flush=True)
    run_server(cmd_args, working_directory)
finally:
    cf_token = ""
    for tunnel in tunnels:
        if tunnel.poll() is None:
            tunnel.terminate()
            try:
                tunnel.wait(timeout=5)
            except subprocess.TimeoutExpired:
                tunnel.kill()
                tunnel.wait(timeout=5)
    save_log()
